# Entrenamiento Red Neuronal Tabular 

## Preparación final de dataset para entrenamiento

Utilizaremos `One-hot Encoding` para evitar que la IA se invente jerarquías matemáticas con los barrios o los tipos de habitación, lo que hacemos es "explotar" esa columna de texto en varias columnas binarias nuevas

De esta forma, la IA entiende perfectamente que son categorías independientes. O lo es (1), o no lo es (0). No hay jerarquías falsas.

Debemos comprobar las columnas categóricas que contengan texto, antes del one-hot encoding

In [2]:
import pandas as pd

df = pd.read_csv('../data/listingV5.csv')

columnas_peligrosas = ['id', 'price_per_person', 'estimated_revenue_l365d', 'estimated_occupancy_l365d']
df_limpio = df.drop(columns=columnas_peligrosas, errors='ignore')

X = df_limpio.drop(columns=['price'])

print("=" * 60)
print("COLUMNAS POR TIPO DE DATO:")
print("=" * 60)
print("\n🔤 OBJECT (texto/categorías):")
obj_cols = X.select_dtypes(include='object').columns.tolist()
for col in obj_cols:
    n_unique = X[col].nunique()
    ejemplo = X[col].dropna().iloc[0] if not X[col].dropna().empty else "N/A"
    print(f"  - {col:<45} | únicos: {n_unique:<5} | ej: {str(ejemplo)[:40]}")

print("\n🔢 NUMÉRICO (int/float):")
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
for col in num_cols:
    print(f"  - {col}")

print("\n✅ BOOL:")
bool_cols = X.select_dtypes(include='bool').columns.tolist()
for col in bool_cols:
    print(f"  - {col}")

print(f"\nTotal columnas: {X.shape[1]}")

COLUMNAS POR TIPO DE DATO:

🔤 OBJECT (texto/categorías):
  - host_response_time                            | únicos: 4     | ej: within an hour
  - neighbourhood_cleansed                        | únicos: 11    | ej: Este
  - property_type                                 | únicos: 39    | ej: Entire rental unit
  - room_type                                     | únicos: 4     | ej: Entire home/apt

🔢 NUMÉRICO (int/float):
  - host_response_rate
  - host_is_superhost
  - host_listings_count
  - latitude
  - longitude
  - accommodates
  - bathrooms
  - bedrooms
  - beds
  - minimum_nights
  - maximum_nights
  - availability_365
  - number_of_reviews
  - number_of_reviews_ltm
  - review_scores_rating
  - review_scores_accuracy
  - review_scores_cleanliness
  - review_scores_checkin
  - review_scores_communication
  - review_scores_location
  - review_scores_value
  - instant_bookable
  - reviews_per_month
  - private_bathroom
  - has_kitchen
  - has_hair_dryer
  - has_iron
  - has_bed_line

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import joblib

# =============================================================================
# PASO 1: CARGA DEL DATASET
# =============================================================================
# Cargamos el CSV final que salió del EDA (ya limpio, sin nulos, sin outliers)
df = pd.read_csv('../data/listingV5.csv')


# =============================================================================
# PASO 2: PURGA DE DATA LEAKAGE (Fuga de Datos)
# =============================================================================
# Data Leakage = incluir en el entrenamiento información que NO existiría
# en el momento real de hacer la predicción. Es el error más grave en ML.
#
# - 'id'                      → Número de serie, no tiene relación causal con el precio
# - 'price_per_person'        → LEAKAGE DIRECTO: se calcula dividiendo 'price' entre
#                               los huéspedes. La red aprendería "trampa" en lugar de
#                               aprender patrones reales del mercado.
# - 'estimated_revenue_l365d' → LEAKAGE DIRECTO: es price * días ocupados. Contiene
#                               el precio de forma implícita.
# - 'estimated_occupancy_l365d'→ Se deriva también de los ingresos estimados.
#
# errors='ignore' evita que el código falle si alguna de estas columnas ya no
# existe en el CSV (por ejemplo si el dataset se actualiza en el futuro).
columnas_peligrosas = ['id', 'price_per_person', 'estimated_revenue_l365d', 'estimated_occupancy_l365d']
df_limpio = df.drop(columns=columnas_peligrosas, errors='ignore')


# =============================================================================
# PASO 3: SEPARAR FEATURES (X) DEL TARGET (y)
# =============================================================================
# X = todo lo que la red neuronal VERÁ para hacer su predicción (las "pistas")
# y = lo que la red neuronal debe APRENDER A PREDECIR (el precio por noche)
#
# Esta separación es fundamental: el modelo nunca debe "ver" el precio
# durante el entrenamiento excepto para calcular el error y ajustar los pesos.
X = df_limpio.drop(columns=['price'])
y = df_limpio['price']


# =============================================================================
# PASO 4: AGRUPAR CATEGORÍAS MINORITARIAS EN 'property_type'
# =============================================================================
# El diagnóstico reveló que 'property_type' tiene 39 categorías únicas.
# Si hacemos One-Hot Encoding directo, generaríamos 38 columnas nuevas,
# muchas de ellas con muy pocos ejemplos (ej: "Yurt" o "Houseboat" con 2 casos).
#
# Problema: una columna con 2 unos en 5000 filas no aporta información útil
# a la red; solo añade ruido y hace el entrenamiento más inestable.
#
# Solución: nos quedamos con los 10 tipos más frecuentes (que representan
# la gran mayoría del dataset) y todo lo demás pasa a llamarse "Other".
# Así controlamos la dimensionalidad sin perder información relevante.
TOP_N_PROPERTY = 10
top_property_types = X['property_type'].value_counts().nlargest(TOP_N_PROPERTY).index
X['property_type'] = X['property_type'].where(
    X['property_type'].isin(top_property_types),  # Si está en el top 10, se mantiene
    other='Other'                                   # Si no, se reemplaza por 'Other'
)

print("Distribución property_type tras agrupación:")
print(X['property_type'].value_counts())


# =============================================================================
# PASO 5: CLASIFICAR COLUMNAS POR TIPO DE TRATAMIENTO
# =============================================================================
# Definimos explícitamente qué columnas son categóricas (texto con categorías)
# para que el ColumnTransformer sepa qué transformación aplicar a cada una.
# El resto de columnas (numéricas) se calcula automáticamente por descarte.
categorical_cols = ['host_response_time', 'neighbourhood_cleansed', 'room_type', 'property_type']
numerical_cols   = [col for col in X.columns if col not in categorical_cols]

print(f"\nColumnas numéricas ({len(numerical_cols)}): {numerical_cols}")


# =============================================================================
# PASO 6: COLUMNTRANSFORMER — EL "DIRECTOR DE ORQUESTA" DEL PREPROCESADO
# =============================================================================
# ColumnTransformer aplica transformaciones DIFERENTES a grupos de columnas
# distintos, todo dentro de un único objeto. Es la forma correcta en Scikit-Learn
# de manejar datasets mixtos (números + texto).
#
# Transformación 1 — StandardScaler para numéricas:
#   Convierte cada columna numérica para que tenga media=0 y desviación=1.
#   Ejemplo: si 'price' va de 20€ a 500€ y 'bathrooms' va de 1 a 5,
#   sin escalar el gradiente "perseguiría" más al precio que a los baños.
#   Con scaling, todas las variables tienen el mismo "peso matemático" inicial.
#
# Transformación 2 — OneHotEncoder para categóricas:
#   Convierte texto en columnas binarias (0 o 1).
#   Ejemplo: room_type='Entire home/apt' → [1, 0, 0, 0]
#            room_type='Private room'    → [0, 1, 0, 0]
#   La red neuronal no puede operar con texto, solo con números.
#
#   · handle_unknown='ignore': Si en producción llega un barrio nuevo que
#     el modelo nunca vio durante el entrenamiento, en lugar de lanzar un
#     error, simplemente pone todos los bits a 0 (categoría desconocida).
#     Crítico para que la API no se caiga en producción.
#
#   · sparse_output=False: Por defecto OneHotEncoder devuelve una matriz
#     "sparse" (dispersa, formato comprimido). PyTorch necesita arrays
#     densos de NumPy, así que forzamos el formato completo.
#
# remainder='drop':
#   Si hubiera alguna columna que no está en numerical_cols ni en
#   categorical_cols (por un despiste), la elimina en lugar de lanzar error.
#   Es la red de seguridad del pipeline.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ],
    remainder='drop'
)


# =============================================================================
# PASO 7: DIVISIÓN EN TRES CONJUNTOS (Train / Validation / Test)
# =============================================================================
# Por qué necesitamos TRES conjuntos y no dos:
#
# · TRAIN (64%): El modelo VE estos datos y ajusta sus pesos con ellos.
#
# · VALIDATION (16%): El modelo NUNCA aprende de estos datos.
#   Se usan DURANTE el entrenamiento para medir si el modelo está
#   generalizando o memorizando (overfitting). También es el conjunto
#   que usará el Early Stopping para decidir cuándo parar.
#
# · TEST (20%): El modelo NUNCA los ha visto en ningún momento.
#   Solo se usan AL FINAL para obtener las métricas definitivas (R², MAE).
#   Simula un apartamento completamente nuevo que llega a la API.
#
# La división se hace en dos pasos:
#   1º → Separamos el Test (20%) del resto (80%)
#   2º → Del 80% restante, separamos Validation (20% de ese 80% = 16% del total)
#        lo que nos deja Train con el 80% de ese 80% = 64% del total.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42   # random_state=42 → reproducibilidad
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.20, random_state=42
)


# =============================================================================
# PASO 8: FIT + TRANSFORM (La regla de oro del preprocesado)
# =============================================================================
# REGLA CRÍTICA: el preprocessor solo puede hacer .fit() sobre datos de Train.
#
# .fit_transform(X_train):
#   · fit()      → Aprende los parámetros: media y desviación de cada columna
#                  numérica, y qué categorías existen en las columnas de texto.
#   · transform()→ Aplica esas transformaciones al propio Train.
#
# .transform(X_val) y .transform(X_test):
#   · Solo aplica los parámetros YA APRENDIDOS del Train. No recalcula nada.
#   · Esto es correcto porque en producción tampoco recalcularíamos: usaríamos
#     los parámetros del entrenamiento para normalizar cada nuevo apartamento.
#
# Si hiciéramos fit() también en Validation/Test, estaríamos "filtrando" 
# información futura al modelo (otro tipo de Data Leakage más sutil).
X_train_scaled = preprocessor.fit_transform(X_train)
X_val_scaled   = preprocessor.transform(X_val)
X_test_scaled  = preprocessor.transform(X_test)


# =============================================================================
# PASO 9: CONVERTIR TARGETS A FLOAT32
# =============================================================================
# PyTorch trabaja internamente con tensores de tipo float32 (32 bits).
# Pandas usa float64 (64 bits) por defecto.
# Convertimos ahora para evitar errores de tipo en el DataLoader de PyTorch.
y_train = y_train.to_numpy(dtype=np.float32)
y_val   = y_val.to_numpy(dtype=np.float32)
y_test  = y_test.to_numpy(dtype=np.float32)


# =============================================================================
# PASO 10: GUARDAR EL PREPROCESSOR PARA PRODUCCIÓN
# =============================================================================

#Por qué no guardar en CSV entonces
#Un CSV solo guarda los datos ya transformados, pero no guarda la receta de cómo transformarlos.

# Con joblib.dump() serializamos el objeto preprocessor a disco.
# Cuando despleguemos la API (FastAPI/Flask), haremos joblib.load() para
# recuperarlo y aplicar exactamente la misma transformación a los datos
# que lleguen de los usuarios, garantizando consistencia total.
import os
os.makedirs('../data', exist_ok=True)
joblib.dump(preprocessor, '../data/preprocessor.pkl')


# =============================================================================
# RESUMEN FINAL
# =============================================================================
print(f"\n✅ Train:      {X_train_scaled.shape}")
print(f"✅ Validation: {X_val_scaled.shape}")
print(f"✅ Test:       {X_test_scaled.shape}")
print(f"\n🧠 Variables de entrada al modelo: {X_train_scaled.shape[1]}")
print("¡Datos listos para PyTorch!")

Distribución property_type tras agrupación:
property_type
Entire rental unit             4129
Private room in rental unit     403
Entire condo                    228
Entire home                     225
Entire serviced apartment       218
Other                           217
Entire loft                     168
Private room in home             87
Entire villa                     42
Entire vacation home             42
Entire townhouse                 37
Name: count, dtype: int64

Columnas numéricas (44): ['host_response_rate', 'host_is_superhost', 'host_listings_count', 'latitude', 'longitude', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'minimum_nights', 'maximum_nights', 'availability_365', 'number_of_reviews', 'number_of_reviews_ltm', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'instant_bookable', 'reviews_per_month', 'private_bathroom', 'has_k

## Activar GPU NVIDIA 

In [1]:
# pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
import torch

print(torch.__version__)               # Debe contener "cu121" o similar, NO "cpu"
print(torch.cuda.is_available())       # Debe imprimir: True
print(torch.cuda.get_device_name(0))   # Debe imprimir el nombre de tu GPU, ej: "NVIDIA GeForce RTX 3060"
device = (
    "cuda"  if torch.cuda.is_available() else
    "mps"   if torch.backends.mps.is_available() else  # Apple Silicon
    "cpu"
)
print(f"Usando dispositivo: {device}")
# Output esperado en tu caso → "Usando dispositivo: cuda"

2.5.1
True
NVIDIA GeForce RTX 4070 Laptop GPU
Usando dispositivo: cuda
